# 03 — Cohort Analysis & Revenue Leakage

**Phase 4 (in-depth analysis).** Tenure-based cohorts, churn-risk matrix,
revenue leakage at 12/24/36-month windows, early-tenure tipping point, and
regional/internet-type disparities.

**Inputs:** `data/processed/clean_customers.csv` (7,043 x 43),
`data/raw/telecom_zipcode_population.csv`.

**Outputs:** cohort tables + `reports/figures/stage4_*.png`.

Denominator rule (matches `sql/views/*`): churn is computed over **existing**
customers only (`customer_status != 'Joined'`); the 454 `Joined` rows are an
acquisition cohort, not churn.


In [ ]:
import matplotlib
matplotlib.use('Agg')
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
REPO = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
CLEAN = os.path.join(REPO, 'data/processed/clean_customers.csv')
POP = os.path.join(REPO, 'data/raw/telecom_zipcode_population.csv')
FIG = os.path.join(REPO, 'reports/figures')
TENURE_ORDER = ['0-12','13-24','25-36','37-48','49-60','60+']
CONTRACT_ORDER = ['Month-to-Month','One Year','Two Year']
BUCKET_ORDER = ['0-3','4-6','7-12','13-24','25-36','37+']
sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi'] = 110
df = pd.read_csv(CLEAN)
df['offer'] = df['offer'].fillna('None')
existing = df[df['customer_status'] != 'Joined'].copy()
print('rows:', len(df), '| existing (non-Joined):', len(existing),
      '| churned:', int(existing['churn'].sum()))
print('overall churn rate: %.4f' % existing['churn'].mean())


## 1. Revenue leakage (12/24/36-month windows)

`revenue_at_risk(h) = Σ monthly_charge × h` over churned customers, for
h ∈ {12,24,36}. This is an **upper bound** (no within-window churn
probability, no discounting). MRR impact baseline = **$137,086.65**.


In [ ]:
# ---- Revenue leakage (12/24/36-month windows) ---------------------------
churned = existing[existing['churn']]
mrr = churned['monthly_charge'].sum()
print('MRR impact: %.2f' % mrr)
rows = []
for contract in CONTRACT_ORDER:
    sub = churned[churned['contract'] == contract]
    m = sub['monthly_charge'].sum()
    rows.append([contract, int(len(sub)), round(m,2), round(m*12,2), round(m*24,2), round(m*36,2)])
rows.append(['(All contracts)', int(len(churned)), round(mrr,2), round(mrr*12,2), round(mrr*24,2), round(mrr*36,2)])
leak = pd.DataFrame(rows, columns=['contract','churned','mrr_impact','risk_12m','risk_24m','risk_36m'])
print(leak.to_string(index=False))
# MRR by contract x tenure_bin (for the stacked-bar figure)
c_t = churned.groupby(['contract','tenure_bin'])['monthly_charge'].sum().unstack()
c_t = c_t.reindex(index=CONTRACT_ORDER, columns=TENURE_ORDER).fillna(0.0)
print(c_t.round(2).to_string())


## 2. Cohort retention curve

% retained (not churned) by `tenure_bin` (6 cohorts).


In [ ]:
# ---- Cohort retention curve --------------------------------------------
ret = existing.groupby('tenure_bin').agg(total=('customer_id','count'), churned=('churn','sum'))
ret['retained'] = ret['total'] - ret['churned']
ret['retention'] = ret['retained'] / ret['total']
ret['churn_rate'] = ret['churned'] / ret['total']
ret = ret.reindex(TENURE_ORDER)
print(ret[['total','churned','retained','retention','churn_rate']].round(4).to_string())


## 3. Churn-risk matrix (contract x tenure_bin)

3 x 6 heatmap of churn rate.


In [ ]:
# ---- Churn-risk matrix (contract x tenure_bin) --------------------------
risk = existing.groupby(['contract','tenure_bin']).agg(total=('customer_id','count'), churned=('churn','sum'))
risk['churn_rate'] = risk['churned'] / risk['total']
risk_matrix = risk['churn_rate'].unstack().reindex(index=CONTRACT_ORDER, columns=TENURE_ORDER)
print('churn rate (%):')
print((risk_matrix*100).round(1).to_string())


## 4. Support-touch tipping point

No support-ticket column exists; proxies are `premium_tech_support` and
`churn_reason ∈ {'Attitude of support person','Attitude of service provider'}`.
Early-tenure buckets: 0-3, 4-6, 7-12, 13-24, 25-36, 37+ months.


In [ ]:
# ---- Support-touch tipping point ----------------------------------------
# Proxies: premium_tech_support + churn_reason in {'Attitude of support person',
#          'Attitude of service provider'} (no support-ticket column exists).
att = churned['churn_reason'].isin(['Attitude of support person','Attitude of service provider'])
print('attitude-driven churned:', int(att.sum()), 'of', len(churned))
existing['bucket'] = pd.cut(existing['tenure_months'], bins=[-1,3,6,12,24,36,1000], labels=BUCKET_ORDER)
vel = existing.groupby('bucket', observed=True).agg(total=('customer_id','count'), churned=('churn','sum'))
vel['churn_rate'] = vel['churned'] / vel['total']
vel = vel.reindex(BUCKET_ORDER)
print(vel.round(4).to_string())
ptb = existing.groupby(['bucket','premium_tech_support'], observed=True).agg(
    total=('customer_id','count'), churned=('churn','sum'))
ptb['churn_rate'] = ptb['churned'] / ptb['total']
print(ptb.round(4).to_string())


## 5. Regional & internet-type disparities

LEFT JOIN on `zip_code`; churn rate by `internet_type`, top zips/cities
(n ≥ 20).


In [ ]:
# ---- Regional & internet-type disparities -------------------------------
pop = pd.read_csv(POP).rename(columns={'Zip Code':'zip_code','Population':'population'})
reg = existing.merge(pop, on='zip_code', how='left')
print('zip coverage: %.3f' % reg['population'].notna().mean())
it = existing.groupby('internet_type').agg(total=('customer_id','count'), churned=('churn','sum'))
it['churn_rate'] = it['churned'] / it['total']
print(it.sort_values('churn_rate', ascending=False).round(4).to_string())
def top_seg(col, nmin, n):
    g = existing.groupby(col).agg(total=('customer_id','count'), churned=('churn','sum'))
    g = g[g['total'] >= nmin]
    g['churn_rate'] = g['churned'] / g['total']
    return g.sort_values('churn_rate', ascending=False).head(n)
print('TOP 5 ZIP (n>=20):')
print(top_seg('zip_code', 20, 5).round(4).to_string())
print('TOP 5 CITY (n>=20):')
print(top_seg('city', 20, 5).round(4).to_string())


## 6. Figures

Writes `stage4_retention_curve.png`, `stage4_risk_matrix_heatmap.png`,
`stage4_revenue_leakage.png`, `stage4_tipping_point.png`,
`stage4_churn_by_zip.png`.


In [ ]:
# ---- Figures ------------------------------------------------------------
os.makedirs(FIG, exist_ok=True)
# retention curve
fig, ax = plt.subplots(figsize=(9,6))
ax.plot(TENURE_ORDER, ret['retention']*100, marker='o', linewidth=2.5, color='#1f77b4')
ax.set_title('Cohort retention curve: % retained by tenure cohort')
ax.set_xlabel('tenure_bin (months)')
ax.set_ylabel('retention rate (%)')
ax.set_ylim(0,105)
for x, r in zip(TENURE_ORDER, ret['retention']*100):
    ax.annotate(f'{r:.1f}%', (x, r), textcoords='offset points', xytext=(0,10), ha='center')
fig.tight_layout(); fig.savefig(os.path.join(FIG,'stage4_retention_curve.png'), bbox_inches='tight'); plt.close(fig)
# risk matrix heatmap
fig, ax = plt.subplots(figsize=(10,6))
sns.heatmap(risk_matrix*100, annot=True, fmt='.1f', cmap='YlOrRd',
            cbar_kws={'label':'churn rate (%)'}, ax=ax, linewidths=0.5)
ax.set_title('Churn-risk matrix: churn rate (%) by contract x tenure_bin')
fig.tight_layout(); fig.savefig(os.path.join(FIG,'stage4_risk_matrix_heatmap.png'), bbox_inches='tight'); plt.close(fig)
# revenue leakage stacked bars (12/24/36)
fig, axes = plt.subplots(1,3,figsize=(18,6), sharey=True)
for ax, h in zip(axes, [12,24,36]):
    stacked = c_t * h
    bottom = np.zeros(len(TENURE_ORDER))
    for c in CONTRACT_ORDER:
        ax.bar(TENURE_ORDER, stacked.loc[c], bottom=bottom, label=c)
        bottom += stacked.loc[c].values
    ax.set_title(f'{h}-month revenue at risk')
    ax.set_xlabel('tenure_bin')
    ax.ticklabel_format(axis='y', style='sci', scilimits=(0,0))
    if ax is axes[0]: ax.set_ylabel('USD at risk')
axes[0].legend(title='contract', fontsize=8, title_fontsize=9)
fig.suptitle('Revenue leakage by tenure cohort x contract (upper-bound, no discounting)', y=1.02)
fig.tight_layout(); fig.savefig(os.path.join(FIG,'stage4_revenue_leakage.png'), bbox_inches='tight'); plt.close(fig)
# tipping point
fig, ax = plt.subplots(figsize=(10,6))
ax.plot(vel.index, vel['churn_rate']*100, marker='o', linewidth=2.5, color='#1f77b4', label='overall churn velocity')
no = (ptb.xs(False, level='premium_tech_support')['churn_rate'].reindex(BUCKET_ORDER)*100)
yes = (ptb.xs(True, level='premium_tech_support')['churn_rate'].reindex(BUCKET_ORDER)*100)
ax.plot(no.index, no.values, marker='s', linestyle='--', color='#d62728', label='without premium_tech_support')
ax.plot(yes.index, yes.values, marker='^', linestyle='--', color='#2ca02c', label='with premium_tech_support')
ax.set_title('Early-tenure churn velocity & support-effect tipping point')
ax.set_xlabel('tenure bucket (months)')
ax.set_ylabel('churn rate (%)')
ax.legend(fontsize=9)
fig.tight_layout(); fig.savefig(os.path.join(FIG,'stage4_tipping_point.png'), bbox_inches='tight'); plt.close(fig)
# top zips / cities
tz = top_seg('zip_code', 20, 5); tc = top_seg('city', 20, 5)
fig, axes = plt.subplots(1,2,figsize=(14,6))
axes[0].barh(tz.index.astype(str)[::-1], tz['churn_rate'][::-1]*100, color='#ff7f0e')
axes[0].set_title('Top 5 high-churn ZIP codes (n>=20)')
axes[0].set_xlabel('churn rate (%)')
axes[1].barh(tc.index[::-1], tc['churn_rate'][::-1]*100, color='#2ca02c')
axes[1].set_title('Top 5 high-churn cities (n>=20)')
axes[1].set_xlabel('churn rate (%)')
fig.tight_layout(); fig.savefig(os.path.join(FIG,'stage4_churn_by_zip.png'), bbox_inches='tight'); plt.close(fig)
print('figures written to', FIG)
